# GPU exact-Hessian dispatch benchmark (issue #126)

This Colab benchmark compares the same issue-#126 branch in two isolated subprocesses:

- **eager**: exact Hessian without `torch.compile`
- **compiled**: exact Hessian with `compile_hessian=True` and `fullgraph=True`

It uses the proven one-day full-workflow collocation workload and reports complete solve time plus first-call and warmed Hessian timings. The first compiled call includes compilation cost; the warmed median measures steady-state dispatch reduction.

Select **Runtime → Change runtime type → GPU** (preferably A100), then run all cells. Both arms must enter IPOPT before timing comparisons are considered valid.

In [ ]:
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys

import torch

REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"
CANDIDATE_REF = "feature/issue-126/reduce-hessian-dispatch"
ROOT = Path("/content/twin4build_hessian_dispatch")
CHECKOUT = ROOT / "candidate"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required.")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        f"git+{REPO_URL}@{CANDIDATE_REF}",
    ],
    check=True,
)

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
subprocess.run(
    [
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        CANDIDATE_REF,
        REPO_URL,
        str(CHECKOUT),
    ],
    check=True,
)

source = (CHECKOUT / "twin4build/estimator/_transcription.py").read_text()
required = ["compile_hessian", "_ss_support", "transform_mode=True"]
missing = [symbol for symbol in required if symbol not in source and symbol != "_ss_support"]
fused_source = (CHECKOUT / "twin4build/systems/utils/fused_statespace_system.py").read_text()
if "_ss_support" not in fused_source:
    missing.append("_ss_support")
if missing:
    raise RuntimeError(f"Candidate checkout is stale; missing {missing}")

props = torch.cuda.get_device_properties(0)
print(
    json.dumps(
        {
            "gpu": props.name,
            "gpu_memory_gb": props.total_memory / 1e9,
            "torch": torch.__version__,
            "python": platform.python_version(),
            "candidate_ref": CANDIDATE_REF,
        },
        indent=2,
    )
)

In [ ]:
# Write the isolated production-workload runner.
RUNNER = ROOT / "run_hessian_arm.py"
RUNNER.write_text(r'''
import datetime
import functools
import importlib.util
import json
import os
from pathlib import Path
import statistics
import subprocess
import sys
import time

import torch

repo = Path(sys.argv[1]).resolve()
label = sys.argv[2]
out_file = Path(sys.argv[3]).resolve()
hours = int(os.environ.get("T4B_BENCH_HOURS", "24"))
maxiter = int(os.environ.get("T4B_BENCH_MAXITER", "20"))
compiled = label == "compiled"

sys.path.insert(0, str(repo))
os.chdir(repo)

from dateutil import tz
import twin4build as tb
import twin4build.estimator._casadi_ipopt as ipopt
import twin4build.examples as examples_package
import twin4build.examples.utils as example_utils

tb._IS_TESTING = True
example_path = Path(examples_package.__file__).parent / "full_workflow_example.py"
spec = importlib.util.spec_from_file_location("_hessian_dispatch_workflow", example_path)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)

STEP_SIZE = 1200
START = datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))


def build_model():
    model = tb.Model(id=f"hessian_dispatch_{label}")
    model.load(
        semantic_model_filename=example_utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=workflow.fcn,
    )
    model.to("cuda", torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc = c["office_temperature_heating_controller"]
    cc = c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.05 / 2),
        (c["office_temperature_sensor"], 0.1 / 2),
        (c["office_damper_position_sensor"], 0.05 / 2),
        (c["office_co2_sensor"], 30 / 2),
    ]


solver_meta = {}
hessian_times = []
original_solve = ipopt.solve_ipopt_constrained


def measured_solve(
    x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jac_rows, jac_cols,
    options=None, *, hess_vals=None, hess_rows=None, hess_cols=None,
    early_stopping=None, print_level=0, quiet=True,
):
    @functools.wraps(hess_vals)
    def timed_hessian(*args):
        torch.cuda.synchronize()
        started = time.perf_counter()
        value = hess_vals(*args)
        torch.cuda.synchronize()
        hessian_times.append(time.perf_counter() - started)
        return value

    torch.cuda.synchronize()
    started = time.perf_counter()
    result = original_solve(
        x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals,
        jac_rows, jac_cols, options=options,
        hess_vals=timed_hessian if hess_vals is not None else None,
        hess_rows=hess_rows, hess_cols=hess_cols,
        early_stopping=early_stopping,
        print_level=print_level, quiet=quiet,
    )
    torch.cuda.synchronize()
    solver_meta.update(
        seconds=time.perf_counter() - started,
        status=str(result.status),
        success=bool(result.success),
        iterations=None if result.nit is None else int(result.nit),
        objective=float(result.fun),
    )
    return result


ipopt.solve_ipopt_constrained = measured_solve
model = build_model()
estimator = tb.Estimator(tb.Simulator(model))
end = START + datetime.timedelta(hours=hours)

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
result = estimator.estimate(
    parameters=build_parameters(model),
    measurements=build_measurements(model),
    start_time=[START],
    end_time=[end],
    step_size=STEP_SIZE,
    n_warmup=20,
    method=("casadi", "ipopt", "ad", "collocation"),
    options={
        "maxiter": maxiter,
        "exact_hessian": True,
        "compile_hessian": compiled,
        "early_stopping": False,
        "boundary_state_init": "rollout",
    },
)
torch.cuda.synchronize()
estimate_seconds = time.perf_counter() - started

if solver_meta.get("iterations", 0) == 0 or solver_meta.get("status") == "Invalid_Number_Detected":
    raise RuntimeError(f"{label} did not enter IPOPT: {solver_meta}")

audit = result.get("transcription_audit", {})
warmed = hessian_times[1:]
row = {
    "arm": label,
    "ref": subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=repo, text=True
    ).strip(),
    "hours": hours,
    "maxiter": maxiter,
    "estimate_seconds": estimate_seconds,
    "solver_seconds": solver_meta["seconds"],
    "status": solver_meta["status"],
    "success": solver_meta["success"],
    "iterations": solver_meta["iterations"],
    "objective": solver_meta["objective"],
    "max_defect": audit.get("max_defect"),
    "peak_cuda_memory_gb": torch.cuda.max_memory_allocated() / 1e9,
    "hessian_calls": len(hessian_times),
    "hessian_total_seconds": sum(hessian_times),
    "hessian_first_seconds": hessian_times[0] if hessian_times else None,
    "hessian_warmed_median_seconds": statistics.median(warmed) if warmed else None,
    "hessian_warmed_min_seconds": min(warmed) if warmed else None,
}
out_file.write_text(json.dumps(row, indent=2))
print(json.dumps(row, indent=2))
''')
print(f"wrote {RUNNER}")

In [ ]:
# Run each mode in a fresh process on the same GPU.
BENCH_HOURS = 24
BENCH_MAXITER = 20

result_files = {
    "eager": ROOT / "eager.json",
    "compiled": ROOT / "compiled.json",
}

for label, result_file in result_files.items():
    print(f"\nRunning {label} exact Hessian ...", flush=True)
    env = os.environ.copy()
    env["T4B_BENCH_HOURS"] = str(BENCH_HOURS)
    env["T4B_BENCH_MAXITER"] = str(BENCH_MAXITER)
    env["PYTHONPATH"] = str(CHECKOUT) + os.pathsep + env.get("PYTHONPATH", "")
    completed = subprocess.run(
        [sys.executable, str(RUNNER), str(CHECKOUT), label, str(result_file)],
        cwd=CHECKOUT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode:
        raise RuntimeError(
            f"{label} failed with exit code {completed.returncode}.\n"
            + "\n".join(completed.stderr.splitlines()[-60:])
        )

rows = [json.loads(path.read_text()) for path in result_files.values()]
print("\nBoth arms completed.")

In [ ]:
import pandas as pd

summary = pd.DataFrame(rows)
eager = summary.loc[summary.arm == "eager"].iloc[0]
compiled = summary.loc[summary.arm == "compiled"].iloc[0]
summary["solver_speedup_vs_eager"] = eager.solver_seconds / summary.solver_seconds
summary["hessian_total_speedup_vs_eager"] = (
    eager.hessian_total_seconds / summary.hessian_total_seconds
)
summary["hessian_warmed_speedup_vs_eager"] = (
    eager.hessian_warmed_median_seconds / summary.hessian_warmed_median_seconds
)

pd.set_option("display.max_columns", None)
display(summary)

print(
    f"Solver speedup: {eager.solver_seconds / compiled.solver_seconds:.3f}x\n"
    f"Total Hessian speedup (includes compile): "
    f"{eager.hessian_total_seconds / compiled.hessian_total_seconds:.3f}x\n"
    f"Warmed median Hessian speedup: "
    f"{eager.hessian_warmed_median_seconds / compiled.hessian_warmed_median_seconds:.3f}x"
)

if eager.status != compiled.status:
    print("WARNING: solver statuses differ; compare timing only after checking objective and defect.")
if abs(eager.objective - compiled.objective) > 1e-6 * max(1.0, abs(eager.objective)):
    print("WARNING: objectives differ materially; do not accept the compiled mode yet.")